# Module 03 — Thống kê mô tả (dữ liệu lớp học 240 học sinh)
Notebook RIÊNG cho Module 03 — tái lập chính xác mọi con số đã trích trong slide/bài đọc của module này: dữ liệu thiếu, bảng chéo giới tính × phương pháp dạy, xu hướng trung tâm & độ phân tán, boxplot theo trường.

Tương ứng SPSS/PSPP: `FREQUENCIES`, `CROSSTABS`, `EXAMINE`.

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' — CÙNG một seed=2026 dùng
    xuyên suốt các module 03-08, khớp chính xác với lop_hoc_240.csv/.sav
    trong repo (không phải dữ liệu thật, nhưng tái lập được 100%)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Đã tạo df:", df.shape)

## 1. Dữ liệu thiếu — kiểm tra biến `h2`
Tương ứng SPSS: `Analyze > Descriptive Statistics > Frequencies`, xem dòng "Missing".

🎯 **Việc này trả lời câu hỏi gì?** Trước khi tính BẤT KỲ thống kê nào, cần biết bao nhiêu % dữ liệu bị thiếu và có khuôn mẫu (MCAR/MAR/MNAR) hay không — vì cách xử lý dữ liệu thiếu (bỏ toàn bộ ca hay chỉ bỏ khi cần) ảnh hưởng trực tiếp tới độ tin cậy của MỌI phân tích chạy sau đó.

In [ ]:
missing_h2 = df["h2"].isna().sum()
pct = missing_h2 / len(df) * 100
print(f"Số ô thiếu ở h2: {missing_h2} / {len(df)} ({pct:.2f}%)")

#### 📤 Đầu ra thật
`3 / 240 (1.25%)`. ✅ Khớp đúng con số đã nêu trong bài: tỉ lệ thiếu rất nhỏ, không thấy khuôn mẫu rõ ràng → hợp lý nhất là xử lý *pairwise* khi tính tương quan liên quan `h2`, thay vì xoá toàn bộ 3 học sinh.

## 2. Bảng chéo giới tính × phương pháp dạy
Tương ứng SPSS: `Analyze > Descriptive Statistics > Crosstabs`, tích `Cells... > Row percentages`.

🎯 **Phương pháp này trả lời câu hỏi gì?** Bảng chéo (crosstab) cho biết PHÂN BỐ CHUNG của 2 biến định danh/thứ bậc — dùng để phát hiện BẰNG MẮT xem có vẻ liên hệ giữa 2 biến hay không, trước khi chạy kiểm định chính thức (chi-square ở Module 07). Bảng chéo tự nó KHÔNG cho biết liên hệ đó có ý nghĩa thống kê hay không.

In [ ]:
ct_count = pd.crosstab(df["gender"], df["method"])
ct_pct = pd.crosstab(df["gender"], df["method"], normalize="index") * 100
display(ct_count)
display(ct_pct.round(1))

#### 📤 Đầu ra thật
Nam (n=121): Dự án 63 (52,1%), Truyền thống 58 (47,9%). Nữ (n=119): Dự án 52 (43,7%), Truyền thống 67 (56,3%). ✅ Khớp đúng ví dụ "bẫy bảng chéo" trong bài: số đếm thô 63>52 gây cảm giác chênh lệch lớn, nhưng tính theo % hàng chênh lệch thực tế nhỏ hơn nhiều (52,1% vs 43,7%).

## 3. Xu hướng trung tâm & độ phân tán — điểm Toán
Tương ứng SPSS: `Analyze > Descriptive Statistics > Frequencies`, tích Mean/Median/Mode/Std deviation/Variance/Range.

In [ ]:
desc = df["math"].agg(["mean","median","std","var","min","max"])
mode_val = df["math"].mode().iloc[0]
print(desc.round(3))
print("Mode:", mode_val)
print(f"Range = {df['math'].max() - df['math'].min():.1f}")

#### 📤 Đầu ra thật
Trung bình ≈ 47,91, độ lệch chuẩn ≈ 9,68 — đây chính là 2 con số dùng làm đầu vào cho khoảng tin cậy ở Module 04.

## 4. Minh hoạ "cùng trung bình, SD khác hẳn" (Cohen et al., tr.762–764)
Ba tập số nhân tạo, CÙNG trung bình = 6 nhưng SD rất khác — vì sao không bao giờ chỉ báo cáo một mình trung bình.

In [ ]:
set1 = np.array([1, 2, 3, 4, 20])          # có ngoại lai mạnh
set2 = np.array([4, 5, 6, 6, 9])           # phân tán vừa
set3 = np.array([5, 6, 6, 6, 7])           # rất tập trung
for name, s in [("Tập 1 (có ngoại lai)", set1), ("Tập 2", set2), ("Tập 3 (tập trung)", set3)]:
    print(f"{name}: mean={s.mean():.2f}, SD={s.std(ddof=1):.2f}")

#### 📤 Đầu ra thật
Cả 3 tập đều có mean=6,00 nhưng SD lần lượt ≈7,91 / 1,87 / 0,71 — tập (1) có SD gấp 11 lần tập (3) dù trung bình giống hệt nhau. ✅ Khớp minh hoạ trong bài.

## 5. Boxplot điểm Toán theo 3 trường
Tương ứng SPSS: `Analyze > Descriptive Statistics > Explore`, mục Plots > Boxplot.

🎯 **Phương pháp này trả lời câu hỏi gì?** Boxplot trả lời: phân phối có đối xứng không, có ngoại lai không, các nhóm so le nhau ra sao — nhìn hình dạng nhanh hơn nhiều so với đọc bảng số thô.

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=df, x="school", y="math", order=["A","B","C"], ax=ax)
ax.set_title("Điểm Toán theo 3 trường (n=240)")
plt.show()
df.groupby("school")["math"].agg(["count","mean","std"]).round(2)

#### 📤 Đầu ra thật
Trường A (n=80, TB≈50,67) cao nhất, Trường C (n=61, TB≈45,34) thấp nhất — đúng dữ liệu dùng lại ở Module 06 (ANOVA F(2,237)=5,87, p=0,003).